# Plot the results of subclone interaction term estimations (error)

This file loads the results of the estimations for $k$ and $m$, subsets them to those that satisfy the biological constraints, and generates Figure 3d.

## Prep

Load needed packages

In [ ]:
%matplotlib widget
from matplotlib import pyplot as plt
import pandas as pd
import numpy as np
import math
import seaborn as sns
import os
from statsmodels.formula import api as smf
from estimator import Estimator
pd.options.mode.chained_assignment = None

Close any figures generated from previous runs

In [ ]:
plt.close("all")

## Define data and user-input parameters

Define the file path for the configuration file (which contains the path to the growth data and estimated growth rates of the subclones), the result path, save path, and names of nude groups.

In [ ]:
groups = ["Grp. B2 nude (80% C1; 20% C11)", "Grp. B3 nude (50% C1; 50% C11)", "Grp. B4 nude (20% C1; 80% C11)"]
result_path = "results/subclone/"
config = "config_subclone.json"
save_mse_file = "figures/nude_km.svg"

Create an Estimator object from the config file

In [ ]:
es = Estimator(config)

Function to get the initial ratio of the subclones based on the group name

In [ ]:
def get_init(group):
    if "A1" in group or "B1" in group: return [1, 0]
    if "A2" in group or "B2" in group: return [0.8, 0.2]
    if "A3" in group or "B3" in group: return [0.5, 0.5]
    if "A4" in group or "B4" in group: return [0.2, 0.8]
    if "A5" in group or "B5" in group: return [0, 1]

## Load and subset the results

Loop through all the result files and create a data frame

In [ ]:
results = []
for fname in [f for f in os.listdir(result_path) if ".csv" in f]:
    temp = pd.read_csv("{}/{}".format(result_path, fname))
    results += [temp]
results = pd.concat(results, ignore_index=True).drop_duplicates()
results

Subset the results to those where all mice have mean squared error in the top 50th percentile 

In [ ]:
# Make a copy of the results data frame so we maintain the original
temp = results.copy()
# For each mouse, determine which pairs have error in the top 50th percentile of errors (of that mouse)
temp["top_quantile"] = results.groupby("id")["error"].transform(lambda x: x < np.quantile(x, 0.5))
# For each k, m pair, determine if every mouse has error in the top 50th percentile
temp1 = temp.groupby(["k", "m"]).apply(lambda x: all(x["top_quantile"])).reset_index()
# Calculate the average values of each parameter combination 
temp2 = results.groupby(["k", "m"])["error"].mean().reset_index()
# Merge the average error data frame and that containing the percentile information
avg_results = pd.merge(temp1, temp2, on=["k", "m"]).drop_duplicates()
avg_results

## Plot (Figure 3d)

In order to make the range of values to plot such that the gradient is visible, set the maximum possible error value to the largest error among the m, k pairs in the top 50th percentiles

In [ ]:
# Find the max error of the points that satisfy the constraints
max_err = avg_results[avg_results[0]]["error"].max()
# If the error is larger than the max error, set to max error 
# (now all points that do not satisfy constraints will be set to the maximum error)
avg_results["error"] = [min(e, max_err) for e in avg_results["error"]]

Pivot the data frame for plotting

In [ ]:
avg_results_pivot = pd.pivot(avg_results, index="k", columns="m", values="error").sort_index(ascending=False)

Plot average error

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=1)
sns.heatmap(avg_results_pivot, ax=ax, yticklabels=[round(j, 3) for j in avg_results_pivot.index], xticklabels=[round(j, 3) for j in avg_results_pivot.columns], square=True, cmap="hot")
# Here we subset the labels so they don't overlap; if you have fewer points, comment these two lines
ax.set_yticks(ax.get_yticks()[::20]) 
ax.set_xticks(ax.get_xticks()[::20])
plt.title("k and m with error in top 50th percentile in all mice")
if save_mse_file:
    plt.savefig(save_mse_file)

## Linear regression between $k$ and $m$ points with low error

In [ ]:
lr_res = smf.ols("k~m", data=avg_results[avg_results[0]]).fit()
lr_res.summary()

Find maximum value of $m$

In [ ]:
avg_results[avg_results[0]]["m"].max()

In [ ]:
plt.close("all")